# Recreate the `cosmdb` Lakebase project

This notebook rebuilds the Lakebase Autoscaling project `cosmdb` from scratch: project, branch, compute, database, roles, Postgres schemas/tables and the Reverse-ETL synced table. It was generated from an inspection of the live project on 2026-11-20.

**Inventory captured from `projects/cosmdb`**

| Layer | Current state |
| --- | --- |
| Project | `cosmdb`, Postgres 18, history retention 7 days, owner josh.rosenberg@databricks.com |
| Project default compute | 2–4 CU, suspend after 24 h |
| Branch | `production` (default, not protected) |
| Endpoint | `primary`, READ_WRITE, autoscale **8–16 CU**, suspend after 24 h, 1 node (no readable secondaries) |
| Databases | `databricks_postgres` (default), **`cosm-transactions`** |
| Roles | `josh.rosenberg@databricks.com` (USER, databricks_superuser) and app SP `50d42876-…` (app **cosm-operations-hub**, CAN_CONNECT_AND_CREATE) |
| Schema `cosm_ops` | `order_items`, `recommender_rules` (50 rows), `training_runs` (2 rows) – owned by the app SP |
| Schema `appkit` | `appkit_cache_entries` – created automatically by the app's AppKit cache |
| Schema `sales` | `transactionspg` – synced table from `jr_febar_catalog.sales.fnb_transaction_items` (SNAPSHOT) |

**Known issue fixed in this rebuild:** the existing synced table `jr_febar_catalog.sales.transactionspg` is in state `SYNCED_TABLE_OFFLINE_FAILED` (0 rows) because it used `transaction_id` as the primary key, but the source has 2,190,644 rows and only 772,978 distinct `transaction_id`s. The composite key `(transaction_id, line_number)` is unique, so this notebook uses it.

**How to use:** set the widgets (defaults target a *new* project `cosmdb-rebuild` so nothing existing is touched), then run top to bottom. Every create step is idempotent – it skips resources that already exist.

In [0]:
%pip install -q --upgrade "databricks-sdk>=0.118.0" "psycopg[binary]>=3.1"
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("project_id", "cosmdb-rebuild", "Target project id")
dbutils.widgets.text("branch_id", "production", "Branch id")
dbutils.widgets.text("endpoint_id", "primary", "Endpoint id")
dbutils.widgets.text("pg_database", "cosm-transactions", "Postgres database")
dbutils.widgets.text("app_sp_client_id", "50d42876-e6fe-452f-ad38-edc4b3987e2b", "App SP client id (blank = skip)")
dbutils.widgets.text("synced_source", "jr_febar_catalog.sales.fnb_transaction_items", "Synced table source (UC)")
dbutils.widgets.text("synced_target", "jr_febar_catalog.sales.transactionspg_rebuild", "Synced table name (UC)")
dbutils.widgets.dropdown("copy_seed_data", "yes", ["yes", "no"], "Copy cosm_ops data from cosmdb")

PROJECT_ID   = dbutils.widgets.get("project_id")
BRANCH_ID    = dbutils.widgets.get("branch_id")
ENDPOINT_ID  = dbutils.widgets.get("endpoint_id")
PG_DATABASE  = dbutils.widgets.get("pg_database")
APP_SP       = dbutils.widgets.get("app_sp_client_id").strip()
SYNC_SOURCE  = dbutils.widgets.get("synced_source")
SYNC_TARGET  = dbutils.widgets.get("synced_target")
COPY_SEED    = dbutils.widgets.get("copy_seed_data") == "yes"
SOURCE_PROJECT = "cosmdb"  # where seed data is copied from

PROJECT  = f"projects/{PROJECT_ID}"
BRANCH   = f"{PROJECT}/branches/{BRANCH_ID}"
ENDPOINT = f"{BRANCH}/endpoints/{ENDPOINT_ID}"

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service import postgres as pg

w = WorkspaceClient()
ME = w.current_user.me().user_name
print(PROJECT, BRANCH, ENDPOINT, PG_DATABASE, "| running as", ME)

In [0]:
# Mirrors cosmdb: Postgres 18, 7-day history retention, project default compute 2-4 CU, suspend after 24h.
# Creating the project auto-provisions the `production` branch and `primary` read-write endpoint.
try:
    project = w.postgres.get_project(name=PROJECT)
    print("Project already exists:", project.name)
except NotFound:
    project = w.postgres.create_project(
        project_id=PROJECT_ID,
        project=pg.Project(spec=pg.ProjectSpec(
            display_name=PROJECT_ID,
            pg_version=18,
            history_retention_duration=pg.Duration(seconds=7 * 24 * 3600),
            default_endpoint_settings=pg.ProjectDefaultEndpointSettings(
                autoscaling_limit_min_cu=2.0,
                autoscaling_limit_max_cu=4.0,
                suspend_timeout_duration=pg.Duration(seconds=24 * 3600),
            ),
        )),
    ).wait()
    print("Created project:", project.name)

for b in w.postgres.list_branches(parent=PROJECT):
    print("branch:", b.name, b.status.current_state, "default=", b.status.default)

In [0]:
# Create the branch only if a non-default branch id was chosen.
try:
    w.postgres.get_branch(name=BRANCH)
except NotFound:
    w.postgres.create_branch(
        parent=PROJECT, branch_id=BRANCH_ID,
        branch=pg.Branch(spec=pg.BranchSpec(source_branch=f"{PROJECT}/branches/production", no_expiry=True)),
    ).wait()
    print("Created branch", BRANCH)

# cosmdb primary endpoint: READ_WRITE, autoscale 8-16 CU, suspend after 24h, single node.
spec = pg.EndpointSpec(
    endpoint_type=pg.EndpointType.ENDPOINT_TYPE_READ_WRITE,
    autoscaling_limit_min_cu=8.0,
    autoscaling_limit_max_cu=16.0,
    suspend_timeout_duration=pg.Duration(seconds=24 * 3600),
)
try:
    w.postgres.get_endpoint(name=ENDPOINT)
    w.postgres.update_endpoint(
        name=ENDPOINT,
        endpoint=pg.Endpoint(spec=spec),
        update_mask=pg.FieldMask(field_mask=[
            "spec.autoscaling_limit_min_cu", "spec.autoscaling_limit_max_cu", "spec.suspend_timeout_duration"]),
    ).wait()
    print("Resized endpoint", ENDPOINT)
except NotFound:
    w.postgres.create_endpoint(parent=BRANCH, endpoint_id=ENDPOINT_ID, endpoint=pg.Endpoint(spec=spec)).wait()
    print("Created endpoint", ENDPOINT)

ep = w.postgres.get_endpoint(name=ENDPOINT)
HOST = ep.status.hosts.host
print(ep.status.current_state, ep.status.autoscaling_limit_min_cu, "-", ep.status.autoscaling_limit_max_cu, "CU |", HOST)

In [0]:
roles = {r.status.postgres_role: r for r in w.postgres.list_roles(parent=BRANCH)}

# The project creator's role is created automatically with databricks_superuser membership.
OWNER_ROLE = roles[ME].name
print("Owner role:", OWNER_ROLE)

# App service principal (cosm-operations-hub). Attaching the database as an app resource also creates
# this role; creating it here lets the GRANTs below run before the app is re-pointed.
if APP_SP and APP_SP not in roles:
    w.postgres.create_role(
        parent=BRANCH,
        role_id=f"dbrx-apps-{APP_SP}",
        role=pg.Role(spec=pg.RoleRoleSpec(
            postgres_role=APP_SP,
            identity_type=pg.RoleIdentityType.SERVICE_PRINCIPAL,
        )),
    ).wait()
    print("Created app SP role", APP_SP)

for r in w.postgres.list_roles(parent=BRANCH):
    print(" ", r.status.postgres_role, r.status.identity_type, r.status.membership_roles)

In [0]:
existing = {d.status.postgres_database: d for d in w.postgres.list_databases(parent=BRANCH)}
if PG_DATABASE in existing:
    print("Database exists:", existing[PG_DATABASE].name)
else:
    db = w.postgres.create_database(
        parent=BRANCH,
        database=pg.Database(spec=pg.DatabaseDatabaseSpec(role=OWNER_ROLE, postgres_database=PG_DATABASE)),
    ).wait()
    print("Created database:", db.name)

for d in w.postgres.list_databases(parent=BRANCH):
    print(" ", d.name, "->", d.status.postgres_database)

In [0]:
import psycopg

def connect(dbname=PG_DATABASE, endpoint=ENDPOINT):
    """Connect with a short-lived Lakebase OAuth token (valid 1 hour)."""
    host = w.postgres.get_endpoint(name=endpoint).status.hosts.host
    token = w.postgres.generate_database_credential(endpoint=endpoint).token
    return psycopg.connect(host=host, dbname=dbname, user=ME, password=token, sslmode="require")

with connect() as conn:
    print(conn.execute("select current_database(), current_user, version()").fetchone())

In [0]:
# DDL reverse-engineered from cosmdb / cosm-transactions (pg_catalog + information_schema).
# `appkit.appkit_cache_entries` is NOT created here: the app's AppKit cache creates it on first start.
# `sales.transactionspg` is NOT created here: the synced table pipeline creates and owns it.
DDL = [
    "CREATE SCHEMA IF NOT EXISTS cosm_ops",
    """
    CREATE TABLE IF NOT EXISTS cosm_ops.order_items (
        id             bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        transaction_id text             NOT NULL,
        item_id        text             NOT NULL,
        quantity       integer          NOT NULL,
        line_total     double precision NOT NULL,
        created_at     timestamptz      NOT NULL DEFAULT now()
    )""",
    "CREATE INDEX IF NOT EXISTS idx_order_items_txn ON cosm_ops.order_items (transaction_id)",
    """
    CREATE TABLE IF NOT EXISTS cosm_ops.recommender_rules (
        id              bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        run_id          text,
        antecedent_id   text,
        antecedent_name text,
        consequent_id   text,
        consequent_name text,
        support         double precision,
        confidence      double precision,
        lift            double precision,
        pair_count      integer,
        trained_at      timestamptz NOT NULL DEFAULT now()
    )""",
    """
    CREATE TABLE IF NOT EXISTS cosm_ops.training_runs (
        run_id            text PRIMARY KEY,
        experiment_id     text,
        baskets           integer,
        distinct_items    integer,
        rules             integer,
        warehouse_baskets integer,
        app_baskets       integer,
        min_support       integer,
        avg_lift          double precision,
        venue             text,
        window_days       integer,
        status            text,
        mlflow_logged     boolean,
        created_at        timestamptz NOT NULL DEFAULT now()
    )""",
    "GRANT ALL ON ALL TABLES IN SCHEMA cosm_ops TO databricks_superuser",
]

with connect() as conn:
    for stmt in DDL:
        conn.execute(stmt)
    print("cosm_ops objects created")

In [0]:
# In cosmdb the app SP owns cosm_ops/appkit. Here the notebook user owns cosm_ops, so grant the SP
# full DML plus CREATE on the database (so AppKit can create its `appkit` schema).
if APP_SP:
    sp = f'"{APP_SP}"'
    grants = [
        f'GRANT CONNECT, CREATE ON DATABASE "{PG_DATABASE}" TO {sp}',
        f"GRANT USAGE, CREATE ON SCHEMA cosm_ops TO {sp}",
        f"GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA cosm_ops TO {sp}",
        f"GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA cosm_ops TO {sp}",
        f"ALTER DEFAULT PRIVILEGES IN SCHEMA cosm_ops GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO {sp}",
    ]
    with connect() as conn:
        for g in grants:
            conn.execute(g)
    print("Granted app SP access on cosm_ops")
else:
    print("No app SP configured - skipped")

In [0]:
# Copies the current contents of cosm_ops (recommender_rules, training_runs, order_items) from the
# source project. Identity values are preserved with OVERRIDING SYSTEM VALUE and sequences are reset.
TABLES = ["training_runs", "recommender_rules", "order_items"]

if COPY_SEED and PROJECT_ID != SOURCE_PROJECT:
    src_ep = f"projects/{SOURCE_PROJECT}/branches/production/endpoints/primary"
    with connect(endpoint=src_ep) as src, connect() as dst:
        src.read_only = True
        for t in TABLES:
            cur = src.execute(f"SELECT * FROM cosm_ops.{t}")
            cols = [d.name for d in cur.description]
            rows = cur.fetchall()
            dst.execute(f"TRUNCATE cosm_ops.{t}")
            if rows:
                override = "OVERRIDING SYSTEM VALUE" if "id" in cols else ""
                placeholders = ", ".join(["%s"] * len(cols))
                with dst.cursor() as c:
                    c.executemany(
                        f"INSERT INTO cosm_ops.{t} ({', '.join(cols)}) {override} VALUES ({placeholders})", rows)
            if "id" in cols:
                dst.execute(f"SELECT setval(pg_get_serial_sequence('cosm_ops.{t}', 'id'), "
                            f"COALESCE((SELECT max(id) FROM cosm_ops.{t}), 0) + 1, false)")
            print(f"cosm_ops.{t}: copied {len(rows)} rows")
else:
    print("Seed copy skipped")

In [0]:
# Syncs the UC Delta table into Postgres as <uc_schema>.<uc_table> (e.g. sales.transactionspg_rebuild).
# Primary key fixed to (transaction_id, line_number): transaction_id alone is not unique in the source,
# which is why the original jr_febar_catalog.sales.transactionspg failed.
src_catalog, src_schema, _ = SYNC_SOURCE.split(".")
try:
    st = w.postgres.get_synced_table(name=f"synced_tables/{SYNC_TARGET}")
    print("Synced table exists:", st.name, st.status.detailed_state)
except NotFound:
    op = w.postgres.create_synced_table(
        synced_table_id=SYNC_TARGET,
        synced_table=pg.SyncedTable(spec=pg.SyncedTableSyncedTableSpec(
            source_table_full_name=SYNC_SOURCE,
            branch=BRANCH,
            postgres_database=PG_DATABASE,
            primary_key_columns=["transaction_id", "line_number"],
            scheduling_policy=pg.SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.SNAPSHOT,
            create_database_objects_if_missing=True,
            new_pipeline_spec=pg.NewPipelineSpec(storage_catalog=src_catalog, storage_schema=src_schema),
        )),
    )
    # The initial snapshot (~2.2M rows) runs in a pipeline; don't block on it here.
    print("Synced table creation started:", op.name())

if APP_SP:
    print("After the first sync completes, grant the app read access, e.g.:")
    print(f'  GRANT USAGE ON SCHEMA {SYNC_TARGET.split(".")[1]} TO "{APP_SP}";')
    print(f'  GRANT SELECT ON {SYNC_TARGET.split(".")[1]}.{SYNC_TARGET.split(".")[2]} TO "{APP_SP}";')

In [0]:
print("Project :", w.postgres.get_project(name=PROJECT).status.pg_version, "(pg_version)")
ep = w.postgres.get_endpoint(name=ENDPOINT)
print("Endpoint:", ep.status.current_state, f"{ep.status.autoscaling_limit_min_cu}-{ep.status.autoscaling_limit_max_cu} CU")
print("Databases:", [d.status.postgres_database for d in w.postgres.list_databases(parent=BRANCH)])
print("Roles    :", [r.status.postgres_role for r in w.postgres.list_roles(parent=BRANCH)])

st = w.postgres.get_synced_table(name=f"synced_tables/{SYNC_TARGET}")
print("Synced   :", SYNC_TARGET, st.status.detailed_state, st.status.message or "")

with connect() as conn:
    rows = conn.execute("""
        SELECT n.nspname, c.relname, c.relkind
        FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname IN ('cosm_ops', 'appkit', 'sales') AND c.relkind IN ('r', 'p')
        ORDER BY 1, 2""").fetchall()
    for schema, table, kind in rows:
        n = conn.execute(f'SELECT count(*) FROM "{schema}"."{table}"').fetchone()[0]
        print(f"  {schema}.{table:<28} rows={n}")

### After the rebuild
* **Re-point the app** `cosm-operations-hub`: change its `postgres` resource to the new branch/database (`projects/<project_id>/branches/production` + `cosm-transactions`) and redeploy. On first start AppKit recreates `appkit.appkit_cache_entries`.
* **Fix the original synced table** (optional): `jr_febar_catalog.sales.transactionspg` in `cosmdb` is still failed with a `transaction_id`-only primary key. Delete it from Catalog Explorer and recreate it with this notebook's cell (set `project_id=cosmdb` and `synced_target=jr_febar_catalog.sales.transactionspg`).
* **Deleting a project** is not done in this notebook on purpose – it permanently removes all branches and data.

In [0]:
# comment for re-commit